# União dos datasets MediaPipe — esquema de publicação v2

Une INCLUDE-50, KSL, MINDS-Libras e LIBRAS-UFOP, com **uma linha por frame**, por leitura
sequencial. Execute em ordem, na raiz do projeto ou em `notebooks`. Requer Python 3.10+ e pandas.
As entradas somam cerca de 12,5 GB; reserve espaço para uma nova cópia dos dados.

O resultado é uma pasta com:

| Arquivo | Conteúdo |
| --- | --- |
| `frames.csv` | 12 campos canônicos + 1.629 coordenadas; 1.641 colunas ao todo. |
| `classes.csv` | Uma linha por classe: rótulo, categoria e identificadores da fonte. |
| `samples.csv` | Uma linha por amostra: arquivo de origem, IDs originais e limites do segmento. |
| `data_dictionary.csv` | Tipo, nulabilidade e significado de cada coluna dos três CSVs. |
| `README.md` | Regras de padronização, chaves, ausências e uso dos metadados. |
| `manifest.json` | Versão do esquema, entradas, contagens e indicação de execução completa/amostra. |

### Decisões de padronização

- `class_id`, `sample_id`, `sequence_id` e `signer_id` são IDs globais com prefixo do dataset.
  Não coexistem com aliases como `class_uid`, `video_uid`, `person` ou `interpreter` no CSV principal.
- `frame` vira `frame_id`. O `category` do MINDS é a classe, não uma categoria temática.
  Rótulos/categorias de fontes diferentes não são fundidos por nome ou número.
- `sample_id` identifica um vídeo ou segmento. `sequence_id` agrupa segmentos da mesma sequência
  do UFOP. Nas demais fontes, sem agrupamento adicional disponível, a sequência coincide com a amostra.
- O `split` do INCLUDE-50 é preservado; nos demais fica vazio. O INCLUDE-50 não recebe um sinalizador
  inventado. Esta união não define um protocolo conjunto de avaliação.
- Os nomes anatômicos dos landmarks do MINDS viram índices, conforme
  `src/preprocessing/landmark_subsets.py` e `src/extraction/mediapipe_extractor.py`.
- Coordenadas finitas mantêm o texto original, sem arredondamento, interpolação ou reescala.
  Campos vazios e `NaN` viram um único marcador de ausência: **campo vazio**. Zero é um valor válido.
- Os flags originais `missing_*` são substituídos por quatro flags com a mesma definição:
  `True` se **qualquer coordenada x/y/z** do respectivo grupo estiver ausente. Não se exporta
  `missing_hand` agregado. Nas fontes, ele tem semânticas diferentes (AND/OR e all/any).
- Metadados de proveniência ficam nas tabelas auxiliares, uma vez por classe/amostra. Os
  identificadores originais nessas tabelas documentam a transformação; não são aliases de treino.

A execução valida chaves de frame, tipos, colunas esperadas, consistência dos metadados e splits
por amostra/sequência. Erros interrompem a exportação, sem deduplicar ou corrigir silenciosamente.
As entradas não são modificadas. A pasta final só aparece quando toda a exportação termina.

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import islice
from urllib.parse import quote
from datetime import datetime, timezone
import csv
import json
import math
import os
import sqlite3
import tempfile

import pandas as pd
from IPython.display import display

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/interim").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Defina ROOT manualmente com a raiz do projeto.")

ROOT = find_project_root()  # Ou: Path("/caminho/para/islr-subset")
SOURCES = {
    "include50": ROOT / "data/interim/include50/include50_mediapipe_with_split.csv",
    "ksl": ROOT / "data/interim/ksl/ksl_mediapipe.csv",
    "minds": ROOT / "data/interim/minds/minds_mediapipe.csv",
    "ufop": ROOT / "data/interim/ufop/ufop_mediapipe.csv",
}

# None processa tudo. Use 100 para um teste; o resultado será marcado como amostra.
MAX_ROWS_PER_DATASET = None
PROGRESS_EVERY = 10_000
output_name = "combined_mediapipe" if MAX_ROWS_PER_DATASET is None else "combined_mediapipe_sample"
OUTPUT_DIR = ROOT / "data/processed" / output_name
# Uma pasta existente nunca é sobrescrita. Escolha outro OUTPUT_DIR para uma nova versão.
print(f"Pasta de saída: {OUTPUT_DIR}")

## 1. Esquema canônico e inspeção dos cabeçalhos

In [ ]:
HAND_LANDMARKS = ['wrist', 'thumb_cmc', 'thumb_mcp', 'thumb_ip', 'thumb_tip', 'index_finger_mcp', 'index_finger_pip', 'index_finger_dip', 'index_finger_tip', 'middle_finger_mcp', 'middle_finger_pip', 'middle_finger_dip', 'middle_finger_tip', 'ring_finger_mcp', 'ring_finger_pip', 'ring_finger_dip', 'ring_finger_tip', 'pinky_mcp', 'pinky_pip', 'pinky_dip', 'pinky_tip']

POSE_LANDMARKS = ['nose', 'left_eye_inner', 'left_eye', 'left_eye_outer', 'right_eye_inner', 'right_eye', 'right_eye_outer', 'left_ear', 'right_ear', 'mouth_left', 'mouth_right', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_pinky', 'right_pinky', 'left_index', 'right_index', 'left_thumb', 'right_thumb', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'left_heel', 'right_heel', 'left_foot_index', 'right_foot_index']

LANDMARK_COLUMNS = [
    f"{part}_{index}_{axis}"
    for part, count in (("face", 468), ("hand_0", 21), ("pose", 33), ("hand_1", 21))
    for index in range(count)
    for axis in "xyz"
]
LANDMARK_SET = set(LANDMARK_COLUMNS)
LANDMARK_ALIASES = {
    f"{part}_{name}_{axis}": f"{part}_{index}_{axis}"
    for part, names in (("hand_0", HAND_LANDMARKS), ("hand_1", HAND_LANDMARKS), ("pose", POSE_LANDMARKS))
    for index, name in enumerate(names)
    for axis in "xyz"
}


FRAME_METADATA = [
    "dataset", "class_id", "sample_id", "sequence_id", "signer_id", "frame_id",
    "source_frame_id", "split", "missing_hand_0", "missing_hand_1", "missing_pose", "missing_face",
]
OUTPUT_COLUMNS = FRAME_METADATA + LANDMARK_COLUMNS
CLASS_COLUMNS = [
    "class_id", "source_class_id", "label", "category_name", "source_category_id",
    "source_class_key", "source_class_id_in_category",
]
SAMPLE_COLUMNS = [
    "sample_id", "source_sample_id", "source_sequence_id", "source_signer_id",
    "source_video_name", "source_video_relpath", "source_start_frame", "source_end_frame",
]
GROUP_COLUMNS = {
    part: [column for column in LANDMARK_COLUMNS if column.startswith(part + "_")]
    for part in ("hand_0", "hand_1", "pose", "face")
}
SOURCE_KEYS = {
    "include50": {"class": "sign_id", "frame": "frame_id", "sample": ("sequence_id",), "signer": None},
    "ksl": {"class": "sign_id", "frame": "frame_id", "sample": ("interpreter", "video_name"), "signer": "interpreter"},
    "minds": {"class": "category", "frame": "frame", "sample": ("person", "video_name"), "signer": "person"},
    "ufop": {"class": "sign_id", "frame": "frame_id", "sample": ("sample_id",), "signer": "participant_id"},
}
SOURCE_METADATA = {
    "include50": "sign_id category sign sign_key sample_id video_name video_relpath sequence_id frame_id missing_hand_0 missing_hand_1 missing_hand missing_pose missing_face split".split(),
    "ksl": "sign sign_id interpreter video_name frame_id missing_hand missing_face".split(),
    "minds": "category video_name frame person missing_hand missing_face".split(),
    "ufop": "sign_id sign_key category_id sign_id_in_category participant_id sample_id sequence_id video_name start_frame end_frame frame_id original_frame_id missing_hand missing_face missing_pose".split(),
}

def inspect_sources(sources):
    schemas = {}
    for dataset, path in sources.items():
        with path.open(encoding="utf-8-sig", newline="") as handle:
            original = next(csv.reader(handle), [])
        normalized = [LANDMARK_ALIASES.get(column, column) for column in original]
        duplicates = [column for column, count in Counter(normalized).items() if count > 1]
        if duplicates:
            raise ValueError(f"{dataset}: colunas duplicadas após padronização: {duplicates}")
        expected = LANDMARK_SET | set(SOURCE_METADATA[dataset])
        if set(normalized) != expected:
            raise ValueError(
                f"{dataset}: esquema inesperado; "
                f"ausentes={sorted(expected - set(normalized))[:10]}, "
                f"extras={sorted(set(normalized) - expected)[:10]}"
            )
        keys = SOURCE_KEYS[dataset]
        required = {keys["class"], keys["frame"], *keys["sample"], "video_name"}
        if keys["signer"]:
            required.add(keys["signer"])
        if dataset == "include50":
            required.add("split")
        if dataset == "ufop":
            required.update(("sequence_id", "start_frame", "end_frame", "original_frame_id"))
        schemas[dataset] = {"original": original, "normalized": normalized, "required": required}
    return schemas

SCHEMAS = inspect_sources(SOURCES)
display(pd.DataFrame([
    {"dataset": dataset, "GB": round(path.stat().st_size / 1e9, 2),
     "colunas_originais": len(SCHEMAS[dataset]["original"]), "coordenadas": len(LANDMARK_COLUMNS)}
    for dataset, path in SOURCES.items()
]))
print(f"frames.csv: {len(OUTPUT_COLUMNS)} colunas ({len(FRAME_METADATA)} metadados + {len(LANDMARK_COLUMNS)} coordenadas).")

## 2. Dicionário de dados

IDs são strings: não remova zeros à esquerda. A chave de `frames.csv` é `(sample_id, frame_id)`;
`class_id` referencia `classes.csv` e `sample_id` referencia `samples.csv`.
`dataset` é mantido explicitamente para filtragem por origem. A repetição de IDs entre frames
é necessária à identificação dos registros; aliases concorrentes foram removidos.

In [ ]:
# campo: (tipo lógico, aceita vazio, descrição)
FRAME_FIELDS = {
    "dataset": ("string", False, "Origem: include50, ksl, minds ou ufop."),
    "class_id": ("string", False, "ID global da classe: dataset::ID original, com escape percentual. Referência a classes.csv."),
    "sample_id": ("string", False, "ID global do vídeo/segmento; referência a samples.csv. Não identifica o sinalizador."),
    "sequence_id": ("string", False, "Grupo de segmentos da mesma sequência no UFOP; nas demais fontes coincide com sample_id. Use para agrupar splits."),
    "signer_id": ("string", True, "ID do sinalizador com prefixo de origem; vazio no INCLUDE-50. Não relaciona pessoas entre datasets."),
    "frame_id": ("integer", False, "Índice não negativo do frame na amostra, copiado de frame_id ou frame. Não é renumerado; lacunas são preservadas."),
    "source_frame_id": ("integer", True, "original_frame_id do UFOP, na convenção da fonte. Vazio nas demais fontes; não se assume a mesma base de frame_id."),
    "split": ("string", True, "train, val ou test no INCLUDE-50; vazio nos demais datasets. Nenhum split novo é criado."),
    "missing_hand_0": ("boolean", False, "True se qualquer coordenada x/y/z da mão 0 (esquerda na extração) estiver ausente; False caso contrário."),
    "missing_hand_1": ("boolean", False, "True se qualquer coordenada x/y/z da mão 1 (direita na extração) estiver ausente; False caso contrário."),
    "missing_pose": ("boolean", False, "True se qualquer coordenada x/y/z da pose estiver ausente; False caso contrário."),
    "missing_face": ("boolean", False, "True se qualquer coordenada x/y/z da face estiver ausente; False caso contrário."),
}
CLASS_FIELDS = {
    "class_id": FRAME_FIELDS["class_id"],
    "source_class_id": ("string", False, "sign_id da fonte, ou category no MINDS; zeros à esquerda preservados."),
    "label": ("string", True, "Texto de sign, disponível no INCLUDE-50 e KSL. Não traduzido; vazio quando não fornecido."),
    "category_name": ("string", True, "Categoria temática de category no INCLUDE-50. O category do MINDS não é copiado para este campo."),
    "source_category_id": ("string", True, "category_id do UFOP; código original, não nome de categoria."),
    "source_class_key": ("string", True, "sign_key original no INCLUDE-50/UFOP, preservado para proveniência."),
    "source_class_id_in_category": ("string", True, "sign_id_in_category do UFOP, preservado para proveniência."),
}
SAMPLE_FIELDS = {
    "sample_id": FRAME_FIELDS["sample_id"],
    "source_sample_id": ("string", True, "sample_id original do INCLUDE-50/UFOP. Pode não ser globalmente único; não usar como chave."),
    "source_sequence_id": ("string", True, "sequence_id original do INCLUDE-50/UFOP, sem prefixo. Vazio quando a fonte não possui esse campo."),
    "source_signer_id": ("string", True, "interpreter do KSL, person do MINDS ou participant_id do UFOP; vazio no INCLUDE-50."),
    "source_video_name": ("string", False, "video_name original, sem inferir se representa o arquivo completo ou segmento."),
    "source_video_relpath": ("string", True, "video_relpath original do INCLUDE-50; vazio nas demais fontes."),
    "source_start_frame": ("integer", True, "start_frame original do UFOP, limite inicial do segmento na convenção da fonte."),
    "source_end_frame": ("integer", True, "end_frame original do UFOP, limite final inclusivo na convenção da fonte."),
}

def data_dictionary():
    records = []
    for table, fields in (("frames.csv", FRAME_FIELDS), ("classes.csv", CLASS_FIELDS), ("samples.csv", SAMPLE_FIELDS)):
        for column, (dtype, nullable, description) in fields.items():
            records.append({"table": table, "column": column, "type": dtype, "nullable": nullable, "description": description})
        if table == "frames.csv":
            for column in LANDMARK_COLUMNS:
                records.append({
                    "table": table, "column": column, "type": "float", "nullable": True,
                    "description": f"Coordenada MediaPipe {column}. Texto numérico finito preservado; vazio representa ausência. Sem reescala ou mudança de referencial.",
                })
    return records

PUBLICATION_README = """# MediaPipe combinado — esquema v2

frames.csv contém uma linha por frame e 1.641 colunas: 12 metadados canônicos e
1.629 coordenadas (543 landmarks x 3 eixos). Consulte data_dictionary.csv para
tipos, nulabilidade e significado de cada coluna. Os CSVs usam UTF-8, vírgula e
cabeçalho único. Ausências são campos vazios; booleanos são True/False.

## Chaves e metadados

Chave de frame: (sample_id, frame_id). classes.csv tem chave class_id;
samples.csv tem chave sample_id. Essas tabelas guardam rótulos e proveniência
uma única vez, sem repetir descrições em todos os frames.

IDs globais têm prefixo do dataset e partes separadas por ::; caracteres
especiais nas partes são escapados por urllib.parse.quote(safe='').
class_id usa sign_id da fonte, exceto MINDS, que usa category.
sample_id usa sequence_id no INCLUDE-50, (interpreter, video_name) no KSL,
(person, video_name) no MINDS e sample_id no UFOP.
sequence_id usa sequence_id da fonte no UFOP; nas demais fontes coincide
com sample_id. signer_id usa interpreter/person/participant_id e fica vazio
no INCLUDE-50. Pessoas e classes de fontes diferentes não são alinhadas.

frame_id preserva o índice local. source_frame_id preserva original_frame_id
do UFOP, sem mudar a base. source_start_frame/source_end_frame preservam os
limites inclusivos da fonte. Não some/subtraia 1 sem verificar a convenção.

## Transformações e valores ausentes

Landmarks anatômicos de mãos/pose do MINDS foram renomeados para índices pela
ordem MediaPipe documentada em src/preprocessing/landmark_subsets.py. A ordem é
face 0..467, hand_0 0..20, pose 0..32, hand_1 0..20; cada índice tem x, y, z.
Valores numéricos finitos não são arredondados, interpolados ou reescalados.
O esquema unifica nomes, sem certificar equivalência dos protocolos de extração
ou dos referenciais das coordenadas entre fontes ou grupos de landmarks.
Campo vazio e NaN (sem distinção de maiúsculas) viram campo vazio. Zero é válido.

missing_hand_0, missing_hand_1, missing_pose e missing_face são recalculados:
True indica QUALQUER coordenada x/y/z ausente no respectivo grupo, incluindo
ausência parcial. Esses campos não medem confiança nem visibilidade/oclusão.
Os flags originais são descartados porque suas definições diferem entre fontes.
Não há missing_hand agregado: ele pode ser derivado explicitamente com OR ou
AND dos flags individuais, conforme o uso, sem impor uma semântica ambígua.

Aliases frame/person/interpreter/participant_id e *_uid não são exportados no
CSV principal. IDs originais, nomes de arquivo e rótulos foram mapeados para
classes.csv e samples.csv. category do MINDS é classe; category do INCLUDE-50
é categoria temática. sign_key é mantido apenas como proveniência da classe.

## Splits e validação

O split original train/val/test do INCLUDE-50 é preservado; demais splits ficam
vazios. Não foi criado um protocolo conjunto. Ao criar splits, agrupe todos os
frames de sample_id e todos os segmentos de sequence_id. Não use sample_id como
proxy de sinalizador. signer_id ausente não constitui uma pessoa adicional.

A exportação rejeita cabeçalhos inesperados, coordenadas não numéricas/infinitas,
frames duplicados, inteiros inválidos, limites UFOP inconsistentes e mudanças de
metadados dentro de uma classe/amostra. Também verifica signer_id/split dentro
de sequence_id. Não descarta frames automaticamente. Duplicidade de chaves é
validada com SQLite temporário, sem guardar todas as coordenadas em memória.

manifest.json informa entradas, contagens e complete_export. Se false, trata-se
somente de um teste limitado por frames e os vídeos podem estar incompletos.
As contagens refletem o conteúdo processado; não comprovam a completude dos
datasets originais. As licenças e versões de extração não são inferidas dos CSVs.

## Leitura

Leia IDs como string, preservando zeros à esquerda. Para coordenadas, deixe o
pandas interpretar campos vazios como NaN. Leia arquivos grandes em chunks.
Associe frames a classes usando class_id e a samples usando sample_id com
validate='many_to_one'; não faça joins por nome do sinal ou IDs locais.
"""

display(pd.DataFrame(data_dictionary()).query("table == 'frames.csv'").head(len(FRAME_METADATA)))

## 3. Normalização e exportação

A escrita é sequencial. Somente os metadados de classes/amostras permanecem em memória;
as chaves de frames são verificadas em um SQLite temporário. Os CSVs auxiliares são escritos
ao final e a pasta temporária é renomeada apenas após todas as validações.

In [ ]:
def make_id(dataset, *parts):
    return "::".join([dataset, *(quote(part, safe="") for part in parts)])

def nonnegative_integer(value, field):
    if not value or not value.isascii() or not value.isdecimal():
        raise ValueError(f"{field}: esperado inteiro não negativo; recebido {value!r}.")
    return int(value)

def normalize_row(dataset, row):
    keys = SOURCE_KEYS[dataset]
    source_class = row[keys["class"]]
    source_signer = row[keys["signer"]] if keys["signer"] else ""
    class_id = make_id(dataset, source_class)
    sample_id = make_id(dataset, *(row[column] for column in keys["sample"]))
    sequence_id = make_id(dataset, row["sequence_id"]) if dataset == "ufop" else sample_id
    frame_id = nonnegative_integer(row[keys["frame"]], keys["frame"])
    source_frame = ""
    if dataset == "ufop":
        source_frame = nonnegative_integer(row["original_frame_id"], "original_frame_id")
        start = nonnegative_integer(row["start_frame"], "start_frame")
        end = nonnegative_integer(row["end_frame"], "end_frame")
        if not start <= source_frame <= end:
            raise ValueError("original_frame_id fora dos limites start_frame/end_frame.")
    split = row.get("split", "")
    if dataset == "include50" and split not in {"train", "val", "test"}:
        raise ValueError(f"split inválido: {split!r}.")
    coordinates = {}
    for column in LANDMARK_COLUMNS:
        value = row[column]
        if value.strip().lower() in {"", "nan"}:
            coordinates[column] = ""
        else:
            if not math.isfinite(float(value)):
                raise ValueError(f"{column}: valor não finito: {value!r}.")
            coordinates[column] = value
    frame = {
        "dataset": dataset, "class_id": class_id, "sample_id": sample_id,
        "sequence_id": sequence_id,
        "signer_id": make_id(dataset, source_signer) if source_signer else "",
        "frame_id": frame_id, "source_frame_id": source_frame, "split": split,
        **{f"missing_{part}": any(coordinates[column] == "" for column in columns)
           for part, columns in GROUP_COLUMNS.items()},
        **coordinates,
    }
    class_record = {
        "class_id": class_id, "source_class_id": source_class,
        "label": row.get("sign", ""),
        "category_name": row["category"] if dataset == "include50" else "",
        "source_category_id": row.get("category_id", ""),
        "source_class_key": row.get("sign_key", ""),
        "source_class_id_in_category": row.get("sign_id_in_category", ""),
    }
    sample_record = {
        "sample_id": sample_id, "source_sample_id": row.get("sample_id", ""),
        "source_sequence_id": row.get("sequence_id", ""), "source_signer_id": source_signer,
        "source_video_name": row["video_name"], "source_video_relpath": row.get("video_relpath", ""),
        "source_start_frame": row.get("start_frame", ""), "source_end_frame": row.get("end_frame", ""),
    }
    return frame, class_record, sample_record

def register_consistent(records, key, value, entity):
    if key in records and records[key] != value:
        raise ValueError(f"Metadados conflitantes para {entity} {key!r}.")
    records.setdefault(key, value)

def write_records(path, columns, records):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, lineterminator="\n")
        writer.writeheader()
        writer.writerows(records)

def merge_datasets(sources, output_dir, max_rows=None, progress_every=10_000):
    if max_rows is not None and (isinstance(max_rows, bool) or not isinstance(max_rows, int) or max_rows <= 0):
        raise ValueError("max_rows deve ser None ou um inteiro positivo.")
    output_dir = Path(output_dir)
    if output_dir.exists():
        raise FileExistsError(f"Pasta já existe: {output_dir}. Escolha outra pasta para esta versão.")
    if not sources:
        raise ValueError("Nenhum dataset informado.")
    schemas = inspect_sources(sources)
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    summary, input_manifest = [], []
    classes, samples, sample_metadata, sequences = {}, {}, {}, {}
    with tempfile.TemporaryDirectory(dir=output_dir.parent, prefix=".mediapipe-stage-") as temp:
        staging_root = Path(temp)
        bundle = staging_root / "bundle"
        bundle.mkdir()
        connection = sqlite3.connect(staging_root / "frame_keys.sqlite")
        try:
            connection.execute("CREATE TABLE frames (sample_id TEXT, frame_id INTEGER, PRIMARY KEY(sample_id, frame_id)) WITHOUT ROWID")
            with (bundle / "frames.csv").open("w", encoding="utf-8", newline="") as destination:
                writer = csv.DictWriter(destination, fieldnames=OUTPUT_COLUMNS, lineterminator="\n")
                writer.writeheader()
                for dataset, path in sources.items():
                    schema = schemas[dataset]
                    stat_before = path.stat()
                    class_ids, sample_ids, signer_ids = set(), set(), set()
                    splits, missing_counts = Counter(), Counter()
                    count = 0
                    print(f"Processando {dataset}...", flush=True)
                    with path.open(encoding="utf-8-sig", newline="") as source:
                        reader = csv.reader(source)
                        if next(reader) != schema["original"]:
                            raise ValueError(f"{dataset}: cabeçalho mudou durante a leitura.")
                        rows = reader if max_rows is None else islice(reader, max_rows)
                        for row_number, values in enumerate(rows, start=2):
                            try:
                                if len(values) != len(schema["normalized"]):
                                    raise ValueError("Quantidade de campos incorreta.")
                                row = dict(zip(schema["normalized"], values))
                                empty = [column for column in schema["required"] if not row[column].strip()]
                                if empty:
                                    raise ValueError(f"Metadados obrigatórios vazios: {empty}.")
                                frame, class_record, sample_record = normalize_row(dataset, row)
                                register_consistent(classes, frame["class_id"], class_record, "classe")
                                register_consistent(samples, frame["sample_id"], sample_record, "amostra")
                                sample_attributes = tuple(frame[column] for column in ("dataset", "class_id", "sequence_id", "signer_id", "split"))
                                register_consistent(sample_metadata, frame["sample_id"], sample_attributes, "amostra")
                                register_consistent(sequences, frame["sequence_id"], (frame["signer_id"], frame["split"]), "sequência")
                                connection.execute("INSERT INTO frames VALUES (?, ?)", (frame["sample_id"], frame["frame_id"]))
                            except sqlite3.IntegrityError as error:
                                raise ValueError(f"{dataset}, registro {row_number}: frame duplicado na amostra.") from error
                            except (ValueError, OverflowError) as error:
                                raise ValueError(f"{dataset}, registro {row_number}: {error}") from error
                            writer.writerow(frame)
                            count += 1
                            class_ids.add(frame["class_id"])
                            sample_ids.add(frame["sample_id"])
                            if frame["signer_id"]:
                                signer_ids.add(frame["signer_id"])
                            splits[frame["split"] or "sem_split"] += 1
                            missing_counts.update({f"missing_{part}": int(frame[f"missing_{part}"]) for part in GROUP_COLUMNS})
                            if progress_every and count % progress_every == 0:
                                print(f"  {dataset}: {count:,} frames", flush=True)
                    if not count:
                        raise ValueError(f"{dataset}: arquivo sem registros.")
                    stat_after = path.stat()
                    if (stat_before.st_size, stat_before.st_mtime_ns) != (stat_after.st_size, stat_after.st_mtime_ns):
                        raise ValueError(f"{dataset}: arquivo alterado durante a leitura.")
                    connection.commit()
                    summary.append({
                        "dataset": dataset, "frames": count, "samples": len(sample_ids),
                        "classes": len(class_ids), "signers": len(signer_ids),
                        **{split: splits[split] for split in ("train", "val", "test", "sem_split")},
                        **missing_counts,
                    })
                    input_manifest.append({"dataset": dataset, "file_name": path.name, "bytes": stat_before.st_size, "frames_read": count})
                    print(f"Concluído: {dataset} — {count:,} frames", flush=True)
            expected_frames = sum(item["frames"] for item in summary)
            if connection.execute("SELECT COUNT(*) FROM frames").fetchone()[0] != expected_frames:
                raise ValueError("A contagem de chaves diverge dos frames escritos.")
        finally:
            connection.close()
        write_records(bundle / "classes.csv", CLASS_COLUMNS, classes.values())
        write_records(bundle / "samples.csv", SAMPLE_COLUMNS, samples.values())
        dictionary = data_dictionary()
        write_records(bundle / "data_dictionary.csv", list(dictionary[0]), dictionary)
        (bundle / "README.md").write_text(PUBLICATION_README, encoding="utf-8")
        manifest = {
            "schema_version": "2.0", "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "complete_export": max_rows is None, "max_rows_per_dataset": max_rows,
            "inputs": input_manifest, "summary": summary,
            "totals": {"frames": expected_frames, "classes": len(classes), "samples": len(samples)},
            "frame_columns": len(OUTPUT_COLUMNS),
            "coordinate_missing_value": "", "missing_flag_rule": "any missing x/y/z coordinate in group",
        }
        (bundle / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        if output_dir.exists():
            raise FileExistsError(f"Pasta de destino criada durante a execução: {output_dir}")
        os.rename(bundle, output_dir)
    return pd.DataFrame(summary)

In [ ]:
summary = merge_datasets(SOURCES, OUTPUT_DIR, max_rows=MAX_ROWS_PER_DATASET, progress_every=PROGRESS_EVERY)
display(summary)
print(f"Total: {summary['frames'].sum():,} frames")
print(f"Pacote salvo em: {OUTPUT_DIR}")
if MAX_ROWS_PER_DATASET is not None:
    print("MODO DE TESTE: vídeos podem estar incompletos. Use MAX_ROWS_PER_DATASET = None para exportar tudo.")

## 4. Conferir a saída e consultar os rótulos

A prévia é limitada. IDs e metadados textuais são lidos como strings para preservar zeros à esquerda.
Os joins são muitos-para-um, sem multiplicar frames. Para ler todas as coordenadas, use
`pd.read_csv(OUTPUT_DIR / "frames.csv", dtype={...}, chunksize=...)`.

In [ ]:
preview = pd.read_csv(OUTPUT_DIR / "frames.csv", nrows=5, dtype="string", keep_default_na=False)
class_lookup = pd.read_csv(OUTPUT_DIR / "classes.csv", dtype="string", keep_default_na=False)
sample_lookup = pd.read_csv(OUTPUT_DIR / "samples.csv", dtype="string", keep_default_na=False)
assert preview.columns.tolist() == OUTPUT_COLUMNS
assert class_lookup["class_id"].is_unique
assert sample_lookup["sample_id"].is_unique
assert (summary["frames"] == summary[["train", "val", "test", "sem_split"]].sum(axis=1)).all()
view = (
    preview[FRAME_METADATA]
    .merge(class_lookup[["class_id", "label", "category_name"]], on="class_id", how="left", validate="many_to_one")
    .merge(sample_lookup[["sample_id", "source_video_name"]], on="sample_id", how="left", validate="many_to_one")
)
assert len(view) == len(preview)
display(view)